# Day 19：Pure-API Function Calling / Tool Use（零框架手写引擎）

> **🎙️ 讲师旁白**：欢迎来到星辰数科《智枢企业级智能体平台》第二阶段第 19 天！今天我们彻底抛开重型 Agent 框架与官方 SDK，使用原生 `httpx` + `ast` 白名单求值器，从零构建标准 OpenAI Function Calling 两轮闭环引擎（`v0.2.0-day19`）。

## 1. 环境初始化与导入 `project_iterations/day_19` 模块
在无 `OPENAI_API_KEY` 的环境下，`ToolCallingAgentLoop` 会自动启用 `httpx.MockTransport` 离线模拟通道，完整还原两轮 HTTP 交互。

In [1]:
import json
import os
import sys
from pathlib import Path

repo_root = Path.cwd()
while not (repo_root / "project_iterations").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
day19_dir = repo_root / "project_iterations" / "day_19"
if str(day19_dir) not in sys.path:
    sys.path.insert(0, str(day19_dir))

os.environ.pop("OPENAI_API_KEY", None)

from zhishu_agent import (
    ToolCallingAgentLoop,
    create_default_registry,
    safe_calculate_math,
)

registry = create_default_registry()
print("已注册工具:", registry.list_tools())
print("Schema 数量:", len(registry.get_schemas()))

已注册工具: ['get_weather', 'calculate_math', 'query_employee_info']
Schema 数量: 3


## 2. 安全防线：AST 白名单数学求值器（严禁 `eval()`）
验证 `safe_calculate_math` 既能正确计算复杂四则运算，又能拦截 `__import__` 等恶意代码注入。

In [2]:
print("正常计算 (128 + 32) * 3 / 4 =", safe_calculate_math("(128 + 32) * 3 / 4"))

try:
    safe_calculate_math("__import__('os').system('echo pwned')")
except ValueError as exc:
    print("成功拦截恶意注入:", exc)

正常计算 (128 + 32) * 3 / 4 = 120
成功拦截恶意注入: 检测到不安全的表达式语法节点: Call


## 3. 端到端 Function Calling 两轮闭环与并行多工具调用
观察单次提问同时触发 `get_weather` 与 `calculate_math` 两个工具，以及消息队列中的 `system -> user -> assistant(tool_calls) -> tool -> assistant` 完整协议流。

In [3]:
agent = ToolCallingAgentLoop()
result = agent.run("帮我查一下北京今天的天气，顺便算一下 (120 + 80) * 3 / 4 等于多少？")

print(f"总交互轮次: {result.rounds} | Token 消耗: {result.total_tokens}")
print("执行工具明细:")
for tc in result.executed_tools:
    print(f"  - {tc['name']}({tc['arguments']}) => {tc['output']}")
print("消息角色序列:", [m["role"] for m in result.messages])
print("最终回复:", result.answer)

总交互轮次: 2 | Token 消耗: 280
执行工具明细:
  - get_weather({"city": "北京", "unit": "celsius"}) => {"city": "北京", "temperature": 18, "unit": "°C", "condition": "晴朗微风", "humidity": "35%"}
  - calculate_math({"expression": "(120 + 80) * 3 / 4"}) => {"expression": "(120 + 80) * 3 / 4", "result": 150}
消息角色序列: ['system', 'user', 'assistant', 'tool', 'tool', 'assistant']
最终回复: 已为您调用工具完成查询：北京当前天气晴朗微风，气温 18°C，湿度 35%；算式 `(120 + 80) * 3 / 4` 的计算结果为 150。


## 4. 员工通讯录查询与未知工具容错演示
演示 `query_employee_info` 工具调用，以及 `ToolRegistry.dispatch()` 面对未知工具名时的优雅降级输出。

In [4]:
emp_result = agent.run("请查询员工陈默的部门与企业邮箱。")
print("员工查询回复:", emp_result.answer)

unknown_dispatch = registry.dispatch("unknown_tool_xyz", "{}")
print("未知工具容错返回:", unknown_dispatch)

员工查询回复: 已为您调用工具完成查询：陈默（EMP-001）隶属AI 应用研发部，担任AI 应用开发工程师，邮箱 chenmo@xingchen.ai。
未知工具容错返回: {"error": "未知工具: unknown_tool_xyz", "available_tools": ["get_weather", "calculate_math", "query_employee_info"]}
